In [1]:
import os
import json
import argparse
import numpy as np
import pandas as pd
import pickle

import tensorflow as tf
from sklearn.model_selection import train_test_split

from build_service_activity_datasets import (
    build_activity_start_dataset,
    save_service_activity_datasets,
)


In [2]:
from SmartHomeHARLib.datasets.casas import Aruba
from SmartHomeHARLib.datasets.casas import Milan
from SmartHomeHARLib.datasets.casas import Cairo

In [3]:
dataset = Aruba()
#dataset = Cairo()
#dataset = Milan()

Load dataset


In [4]:
df = dataset.df

In [5]:
df

,datetime,sensor,value,activity,activityState
0,2010-11-04 00:03:50.209589,M003,ON,Sleeping,begin
1,2010-11-04 00:03:57.399391,M003,OFF,Sleeping,NaN
2,2010-11-04 00:15:08.984841,T002,21.5,Sleeping,NaN
3,2010-11-04 00:30:19.185547,T003,21,Sleeping,NaN
4,2010-11-04 00:30:19.385336,T004,21,Sleeping,NaN
...,...,...,...,...,...
1713123,2011-06-11 23:42:59.285070,T002,25.5,Sleeping,NaN
1713124,2011-06-11 23:48:02.888409,T001,23.5,Sleeping,NaN
1713125,2011-06-11 23:48:02.988798,T002,25,Sleeping,NaN
1713126,2011-06-11 23:53:06.429200,T002,25.5,Sleeping,NaN


In [6]:
service_activity_sequences = build_activity_start_dataset(dataset.name.lower(), dataset.df)
service_activity_sequences.head()


,dataset,activity_label,service_activity_label,activity_start_time,activity_start_hour,day_of_week,time_slot_label,recommended_appliance,recommended_appliance_candidates
0,aruba,Sleeping,휴식/수면,2010-11-04 00:03:50.209589,0,목요일,새벽,추천없음,추천없음
1,aruba,Bed_to_Toilet,기타,2010-11-04 05:40:51.303739,5,목요일,아침,추천없음,추천없음
2,aruba,Sleeping,휴식/수면,2010-11-04 05:43:45.732400,5,목요일,아침,추천없음,추천없음
3,aruba,Meal_Preparation,식사준비,2010-11-04 08:11:09.966157,8,목요일,아침,공기청정기,공기청정기
4,aruba,Meal_Preparation,식사준비,2010-11-04 08:33:52.929406,8,목요일,아침,공기청정기,공기청정기


In [7]:
print("service_activity_label class distribution")
print(service_activity_sequences['service_activity_label'].value_counts(dropna=False).sort_index())

print("\ntime_slot_label distribution")
print(service_activity_sequences['time_slot_label'].value_counts(dropna=False).sort_index())


service_activity_label class distribution
service_activity_label
귀가        427
기타        333
설거지        64
식사        255
식사준비     1596
외출        427
청소/정리      33
휴식/수면    3305
Name: count, dtype: int64

time_slot_label distribution
time_slot_label
밤      598
새벽     329
아침    1076
오전    1036
오후    1529
저녁    1271
점심     601
Name: count, dtype: int64


In [8]:
output_paths = save_service_activity_datasets(service_activity_sequences, "datasets")
output_paths


{'processed_service_activity_sequences': 'datasets\\processed_service_activity_sequences.csv',
 'activity_start_time_dataset': 'datasets\\activity_start_time_dataset.csv',
 'appliance_recommendation_dataset': 'datasets\\appliance_recommendation_dataset.csv'}

In [9]:
df['hour'] = df['datetime'].dt.hour.astype(int) + 1
#df['hour'] = df['hour'].replace(to_replace = 0, value = 24)
df['minutes'] = df['datetime'].dt.minute.astype(int) + 1
df['secondes'] = df['datetime'].dt.second.astype(int) + 1
df['weekday'] = df['datetime'].dt.dayofweek.astype(int) + 1



In [10]:
df[:60]

,datetime,sensor,value,activity,activityState,hour,minutes,secondes,weekday
0,2010-11-04 00:03:50.209589,M003,ON,Sleeping,begin,1,4,51,4
1,2010-11-04 00:03:57.399391,M003,OFF,Sleeping,NaN,1,4,58,4
2,2010-11-04 00:15:08.984841,T002,21.5,Sleeping,NaN,1,16,9,4
3,2010-11-04 00:30:19.185547,T003,21,Sleeping,NaN,1,31,20,4
4,2010-11-04 00:30:19.385336,T004,21,Sleeping,NaN,1,31,20,4
5,2010-11-04 00:35:22.245870,T005,20.5,Sleeping,NaN,1,36,23,4
6,2010-11-04 00:40:25.428962,T005,21,Sleeping,NaN,1,41,26,4
7,2010-11-04 00:45:28.658171,T005,20.5,Sleeping,NaN,1,46,29,4
8,2010-11-04 01:05:42.269469,T001,20,Sleeping,NaN,2,6,43,4
9,2010-11-04 01:15:48.936777,T002,21,Sleeping,NaN,2,16,49,4


In [11]:
#df['week'] = df['datetime'].apply(lambda x: x.weekofyear if x.weekday() <= 6 else x.weekofyear+1)
df['week'] = df['datetime'].dt.isocalendar().week.astype(int)

In [12]:
df[29348:29398
]

,datetime,sensor,value,activity,activityState,hour,minutes,secondes,weekday,week
29348,2010-11-07 23:50:54.709651,M003,ON,Other,NaN,24,51,55,7,44
29349,2010-11-07 23:50:58.295552,M003,OFF,Other,NaN,24,51,59,7,44
29350,2010-11-07 23:51:04.643101,M003,ON,Other,NaN,24,52,5,7,44
29351,2010-11-07 23:51:07.555403,M008,ON,Other,NaN,24,52,8,7,44
29352,2010-11-07 23:51:07.625005,M003,OFF,Other,NaN,24,52,8,7,44
29353,2010-11-07 23:51:07.718853,M020,ON,Other,NaN,24,52,8,7,44
29354,2010-11-07 23:51:10.531130,M008,OFF,Other,NaN,24,52,11,7,44
29355,2010-11-07 23:51:10.923748,M020,OFF,Other,NaN,24,52,11,7,44
29356,2010-11-07 23:51:13.556385,M020,ON,Other,NaN,24,52,14,7,44
29357,2010-11-07 23:51:14.562280,M008,ON,Other,NaN,24,52,15,7,44


In [13]:
def split_dataset_into_week(df):

    chunks = []

    transitionIndex = df.week.ne(df.week.shift())
    ii = np.where(transitionIndex == True)[0]
    

    for i, end in enumerate(ii):
        if i > 0:
            start = ii[i - 1]

            # activitySeq = self.df[start:end]
            if(len(df[start:end])>1):
                chunks.append(df[start:end])

    # lastActivitySeq = self.df[end:]

    if(len(df[end:])>1):
        chunks.append(df[end:])

    return chunks

In [14]:
weeks = split_dataset_into_week(df)

In [15]:
train_data, test_data = train_test_split(weeks, test_size=0.3, random_state=42)

In [16]:
# Convert the training and testing sets back into a list of DataFrames
train_data = [pd.DataFrame(train_data[i]) for i in range(len(train_data))]
test_data = [pd.DataFrame(test_data[i]) for i in range(len(test_data))]

In [17]:
train_data[0]

,datetime,sensor,value,activity,activityState,hour,minutes,secondes,weekday,week
173681,2010-11-29 00:00:05.424831,M009,OFF,Relax,NaN,1,1,6,1,48
173682,2010-11-29 00:03:34.011945,M009,ON,Relax,NaN,1,4,35,1,48
173683,2010-11-29 00:03:35.830651,M009,OFF,Relax,NaN,1,4,36,1,48
173684,2010-11-29 00:04:42.650997,T004,21,Relax,NaN,1,5,43,1,48
173685,2010-11-29 00:14:11.411195,M009,ON,Relax,NaN,1,15,12,1,48
...,...,...,...,...,...,...,...,...,...,...
225428,2010-12-05 23:57:18.165090,M002,ON,Sleeping,NaN,24,58,19,7,48
225429,2010-12-05 23:57:22.343125,M002,OFF,Sleeping,NaN,24,58,23,7,48
225430,2010-12-05 23:57:23.331223,M003,OFF,Sleeping,NaN,24,58,24,7,48
225431,2010-12-05 23:57:56.445867,M002,ON,Sleeping,NaN,24,58,57,7,48


In [18]:
dataset.name.lower()

'aruba'

In [19]:
# Save the list of DataFrames to a pickle file
with open("datasets/"+dataset.name.lower()+'_train_data_time.pickle', 'wb') as f:
    pickle.dump(train_data, f)

In [20]:
# Save the list of DataFrames to a pickle file
with open("datasets/"+dataset.name.lower()+'_test_data_time.pickle', 'wb') as f:
    pickle.dump(test_data, f)

In [21]:
# Load the list of DataFrames from the pickle file
with open("datasets/"+dataset.name.lower()+'_train_data_time.pickle', 'rb') as f:
    df_list = pickle.load(f)